# 05 Backtest and Regime Analysis

This notebook runs `scripts/evaluate_backtest.py`, which reads the calibrated model predictions from notebook 04 and regenerates regime metrics, long/flat backtests, robustness checks, failure cases, and continuous return-model diagnostics.


## Source Code Note

The full reproducible evaluation implementation lives in `scripts/evaluate_backtest.py`. This notebook calls that script, then displays regime metrics, backtest tables, robustness checks, failure cases, and return-model diagnostics.


## 1. Run Evaluation Pipeline

The main test-set backtest uses thresholds selected on the validation split. Expanding-window OOS results are still reported for regime discussion, using a default 0.50 threshold to avoid future validation leakage into early years.


In [ ]:
from pathlib import Path
import runpy

import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data" / "processed"
FIG = ROOT / "figures"
REPORT = ROOT / "report"

runpy.run_path(str(ROOT / "scripts" / "evaluate_backtest.py"), run_name="__main__")


## 2. Load Outputs


In [ ]:
regime_metrics = pd.read_csv(DATA / "model_regime_metrics.csv")
backtest_summary = pd.read_csv(DATA / "backtest_performance_summary.csv")
backtest_regime = pd.read_csv(DATA / "backtest_regime_summary.csv")
robustness = pd.read_csv(DATA / "robustness_threshold_cost_summary.csv")
failure_cases = pd.read_csv(DATA / "model_failure_cases.csv", parse_dates=["Date"])
return_metrics = pd.read_csv(DATA / "return_model_metrics_summary.csv")

for name, df in [
    ("regime_metrics", regime_metrics),
    ("backtest_summary", backtest_summary),
    ("backtest_regime", backtest_regime),
    ("robustness", robustness),
    ("failure_cases", failure_cases),
    ("return_metrics", return_metrics),
]:
    print(f"{name:24s} {df.shape}")


## 3. Test Classification, All Features

These are the held-out 2022-2024 results using validation-selected thresholds.


In [ ]:
test_regime = regime_metrics[
    (regime_metrics["split"] == "test")
    & (regime_metrics["feature_set"] == "all")
    & (regime_metrics["model_name"].isin(["Logit", "XGB"]))
].copy()

display(
    test_regime[["model_name", "regime", "n", "threshold", "up_day_rate", "accuracy", "balanced_accuracy", "auc", "f1", "brier"]].round(4)
)


## 4. Long/Flat Backtest on Test Set

The strategy is long when calibrated predicted up probability is above the validation-selected threshold, otherwise flat. Transaction cost is 1 basis point per signal change.


In [ ]:
test_backtest = backtest_summary[
    (backtest_summary["split"] == "test")
    & (backtest_summary["feature_set"] == "all")
    & (backtest_summary["model_name"].isin(["Logit", "XGB"]))
].copy()

display(
    test_backtest[[
        "model_name", "threshold", "strategy_cumulative_return", "buy_hold_cumulative_return",
        "strategy_sharpe", "strategy_max_drawdown", "exposure", "trades"
    ]].round(4)
)

display(Image(filename=str(FIG / "fig6_backtest_equity_curves.png")))


## 5. Expanding-Window Regime Check

This keeps the original regime-analysis idea: each year is predicted using only earlier years.


In [ ]:
oos_regime = regime_metrics[
    (regime_metrics["split"] == "oos_expanding")
    & (regime_metrics["feature_set"] == "all")
    & (regime_metrics["model_name"].isin(["Logit", "XGB"]))
].copy()

display(
    oos_regime[["model_name", "regime", "n", "threshold", "accuracy", "balanced_accuracy", "auc", "f1", "up_day_rate"]].round(4)
)

display(Image(filename=str(FIG / "fig7_regime_backtest_returns.png")))


## 6. Threshold and Cost Robustness

Robustness varies threshold and transaction cost; the main test report still uses the validation-selected thresholds.


In [ ]:
display(
    robustness.sort_values("strategy_cumulative_return", ascending=False)[
        ["split", "model_name", "threshold", "transaction_cost", "strategy_cumulative_return", "strategy_sharpe", "strategy_max_drawdown", "exposure", "trades"]
    ].head(12).round(4)
)

display(Image(filename=str(FIG / "fig8_backtest_robustness.png")))


## 7. Failure Cases

These rows are useful for final-report discussion: false long losses, missed rallies, and high-confidence classification errors.


In [ ]:
display(
    failure_cases[[
        "failure_type", "Date", "regime", "model_name", "y_pred_proba", "threshold",
        "return_next_day", "headline_count", "vix", "daily_text_snippet"
    ]].head(15).round(4)
)


## 8. Continuous Return Modeling Supplement

This fills the proposal requirement to also examine continuous next-day returns.


In [ ]:
test_return_metrics = return_metrics[return_metrics["split"] == "test"].copy()
display(
    test_return_metrics.sort_values("information_coefficient", ascending=False)[
        ["model", "feature_set", "rmse", "mae", "r2", "information_coefficient", "direction_accuracy"]
    ].head(12).round(5)
)

display(Image(filename=str(FIG / "fig9_return_model_predictions.png")))


## 9. Evaluation Takeaways

- Balanced weighting, calibrated probabilities, and validation-selected thresholds improve the methodology.
- Test-set gains are still small, so the conclusion should remain cautious.
- The honest final story is limited and unstable signal, not a reliable trading edge.
